In [49]:
!rm -rf /content/Repo
!git clone https://github.com/Krishuang-1116/ai-healthcare-final-project.git /content/Repo


Cloning into '/content/Repo'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 50 (delta 20), reused 34 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 1009.67 KiB | 7.48 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [50]:
import os
import sys
from pathlib import Path

# For Colab after cloning the repo
PROJECT_ROOT = Path("/content/Repo")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
METRICS_DIR = OUTPUTS_DIR / "metrics"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

In [51]:
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR =", DATA_DIR)
print("OUTPUTS_DIR =", OUTPUTS_DIR)

PROJECT_ROOT = /content/Repo
DATA_DIR = /content/Repo/data
OUTPUTS_DIR = /content/Repo/outputs


In [52]:
# Colab installs
!pip install -q pandas numpy scikit-learn xgboost interpret torch jupyter


In [53]:
from src.data_utils import *
from src.metrics_utils import *
from src.results_utils import *


# Test run MLPClassifierWrapper on fake data

In [54]:
from src.models_deep import MLPClassifierWrapper
import numpy as np

# tiny fake data smoke test
X_fake = np.random.randn(100, 10)
y_fake = np.random.randint(0, 2, size=100)

mlp = MLPClassifierWrapper(
    input_dim=X_fake.shape[1],
    hidden_dim=32,
    epochs=3,
    batch_size=16
)

mlp.fit(X_fake, y_fake)

probs = mlp.predict_proba(X_fake)
preds = mlp.predict(X_fake)

print("probs shape:", probs.shape)
print("preds shape:", preds.shape)
print("first 5 probs:", probs[:5])
print("first 5 preds:", preds[:5])


probs shape: (100, 2)
preds shape: (100,)
first 5 probs: [[0.508198   0.491802  ]
 [0.4232238  0.5767762 ]
 [0.4733538  0.5266462 ]
 [0.50668365 0.49331635]
 [0.44873893 0.55126107]]
first 5 preds: [0 1 1 0 1]


In [55]:
from src.data_utils import load_data, basic_data_check,get_feature_target
df_path = DATA_DIR / "final_project_data.csv"
df = load_data(df_path)
basic_data_check(df)

X, y, ids = get_feature_target(df)

print('\nX shape:', X.shape)
print('y shape:', y.shape)
print('ids shape:', ids.shape)

Shape: (12424, 26)
Columns: ['subject_id', 'hadm_id', 'stay_id', 'intime', 'heart_rate_mean', 'sbp_mean', 'dbp_mean', 'mbp_mean', 'resp_rate_mean', 'temperature_mean', 'spo2_mean', 'wbc_min', 'wbc_max', 'platelets_min', 'hemoglobin_min', 'creatinine_max', 'sodium_min', 'sodium_max', 'glucose_max', 'albumin_min', 'lactate_max', 'ph_min', 'po2_min', 'pco2_max', 'sofa', 'hospital_expire_flag']

Target distribution:
hospital_expire_flag
0    9698
1    2726
Name: count, dtype: int64

Missing values:
albumin_min             47.883129
lactate_max             32.807469
ph_min                  21.555055
pco2_max                21.555055
po2_min                 21.547006
temperature_mean         1.827109
wbc_min                  0.861236
wbc_max                  0.861236
glucose_max              0.853187
platelets_min            0.845138
hemoglobin_min           0.829041
sodium_min               0.804894
sodium_max               0.804894
creatinine_max           0.748551
mbp_mean                

In [56]:
from src.cv_runner import run_cv_model
import src.models_deep
from src.models_classical import build_missingness_indicator

In [57]:
mlp_metrics_df, mlp_preds_df, mlp_params_df = run_cv_model(
    X=X,
    y=y,
    ids=ids,
    model_name="mlp",
    fit_predict_fn=fit_predict_mlp
)


ValueError: NaN loss encountered during MLP training.